# పాఠం 13 - ఏజెంట్ మెమరీ


## సెటప్

ఈ నోటుబుక్ **Microsoft Agent Framework** (MAF) ఉపయోగించి **స్థిరమెమైన మెమ్మరీ** తో ఒక ట్రావెల్ బుకింగ్ ఏజెంట్ ఎలా నిర్మించాలో చూపిస్తుంది.

ఏజెంట్ మెమ్మరీ యొక్క వివిధ రకాలైన — వర్కింగ్, షార్ట్-టర్మ్, మరియు లాంగ్-టర్మ్ — ఏజెంట్ సంభాషణల మధ్య సమాచారం ఎలా నిలుపుకుంటుందో మరియు ఉపయోగిస్తుందో మీకు నేర్పుతుంది.

**ముందస్తు అవసరాలు:**
- ఒక Azure AI Foundry ప్రాజెక్ట్, దానిలో ఒక చాట్ మోడల్ (ఉదా: `gpt-4o-mini`) డిప్లాయ్ చేయబడింది.
- Azure CLIలో లాగిన్ అయి ఉండాలి — మీ టెర్మినల్‌లో `az login` ను నడపండి.
- `AZURE_AI_PROJECT_ENDPOINT` — మీ Azure AI Foundry ప్రాజెక్టు ఎండ్‌పాయింట్.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — మీ డిప్లాయ్ చేసిన మోడల్ యొక్క పేరు.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## ఏజెంట్ మెమొరీ రకాలు

AI ఏజెంట్లు విభిన్న రకాల మెమొరీలను ఉపయోగించవచ్చు, ప్రతి ఒక్కటి వేరే ఉద్దేశ్యాన్ని అందిస్తుంది:

### వర్కింగ్ మెమొరీ  
సంభాషణ తాడు స్వయంగా — ఒకే సెషన్‌లో మార్పిడి చేసిన సందేశాలు. ఏజెంట్ ఒకే తాడులో మొదటివిగా వచ్చిన సందేశాలకు తిరిగి చూడగలదు సరిహద్దును నిలుపుకునేందుకు. MAFలో ఇది **`agent.create_session()`** ద్వారా సృష్టించబడుతుంది, ఇది ఒక `AgentSession` ని ఇస్తుంది.

### షార్ట్-టర్మ్ మెమొరీ  
ఒక కర్తవ్యం లేదా సెషన్ కాలంలో నిలిచే సమాచారము కానీ శాశ్వతంగా నిల్వ చేయబడదు. ఉదాహరణకు, ఏజెంట్ బహుళ-తిరుగుడు ప్రణాళిక సంభాషణలో నిజాలనూ సేకరించి వాటిని ఉపయోగించి తుది ప్రయాణక్రమాన్ని రూపొందించవచ్చు.

### లాంగ్-టర్మ్ మెమొరీ  
**సేశన్ల మధ్య** నిలిచే అభిరుచి మరియు నిజాలు. తిరిగి వచ్చిన వినియోగదారు తన ఆహార పరిమితులు లేదా ప్రయాణ శైలిని మళ్లీ చెప్పాల్సిన అవసరం ఉండదు. లాంగ్-టర్మ్ మెమొరీ సాధారణంగా బయటి నిల్వ ద్వారా మద్దతు పొందుతుంది — ఒక డేటాబేస్, ఫైరు లేదా వెక్టార్ సూచిక — మరియు టూల్స్ ద్వారా ఏజెంట్ వద్దకి తీసుకువస్తుంది.


## సెషన్లతో వర్కింగ్ మెమోరీ

మెమోరీ యొక్క అతి సాంప్రదాయ రూపం సంభాషణ సెషన్. మీరు అదే సెషన్ (`agent.create_session()` ద్వారా సృష్టించబడిన)ని వరుసగా `agent.run()` కాల్స్‌కు పాస్ చేస్తే, ఏజెంట్ ఆ సంభాషణ పూర్తి చరిత్రను చూస్తుంది మరియు ముందటి వివరాలను గుర్తు చేసుకోగలదు.

వార్కింగ్ మెమోరీని ప్రదర్శించడానికి ఒక ట్రావెల్ ఏజెంట్ సృష్టిద్దాం.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

ఏజెంట్ బడ్జెట్‌ను సరిగ్గా గుర్తుంచుకున్నది ఎందుకంటే రెండు సందేశాలు ఒకే సెషన్‌ను పంచుకుంటున్నాయి. ఇది **పనిచేసే జ్ఞాపకశక్తి** — ఇది సెషన్ జీవితకాలం కే ఉన్నది.

### కొత్త థ్రెడ్‌తో ఏమి జరుగుతుంది?

మనం ఒక **కొత్త** సెషన్ సృష్టిస్తే, ఏజెంట్ కి మునుపటి సంభాషణ జ్ఞాపకం లేదు:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## దీర్ఘకాలిక జ్ఞాపక నమూనా

వినియోగదారు ప్రాధాన్యతలను **సెషన్ల మధ్య** గుర్తుంచుకోవడానికి, సంభాషణ థ్రెడ్ బయట ఉండే ఒక శాశ్వత నిల్వ అవసరం. ఏజెంట్ ఈ నిల్వను సేవ్ చేయడానికి మరియు సమాచారాన్ని తీసుకురావడానికి కాల్ చేయగల **టూల్స్** ద్వారా యాక్సెస్ చేస్తుంది.

క్రింద మేము ఒక సులభమైన ఇన్-మెమరీ ప్రాధాన్యత నిల్వను అమలు చేస్తాము (ఉత్పత్తిలో మీరు దీన్ని డేటాబేస్ లేదా వెక్టర్ సూచికతో మద్దతు ఇస్తారు) మరియు ఏజెంట్ ఉపయోగించగల టూల్స్ గా దీన్ని లభ్యం చేస్తాము.

### నిర్మాణం
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### సందర్భం 1 — మొదటి సారి వినియోగదారు వార్షికోత్సవ పర్యటన బుక్ చేస్తుంది

సారా మొదటి సారి సందర్శిస్తుంది. ఏజెంట్ ఆమె అభిరుచులను సాధనాల ద్వారా నిల్వ చేసి, వాటిని హోటల్‌లను సిఫారసు చేయడానికి ఉపయోగించాలి.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### సన్నివేశం 2 — సారా వారం లేటుగా తిరిగి వస్తుంది

సారా ఒక **కొత్త థ్రెడ్** ప్రారంభిస్తుంది (కొత్త సెషన్‌ను అనుకరిస్తోంది). పని స్మృతి ఖాళీగా ఉంటుంది, కానీ దీర్ఘకాలిక ప్రాధాన్యత దుకాణంలో ఆమె సమాచారం ఇంకా ఉంటుంది. ఏజెంట్ దాన్ని తిరిగి తెచ్చుకుని వ్యక్తిగతీకరించిన సిఫార్సులను అందించాలి.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## సారాంశం

ఈ పాఠంలో మీరు మూడు రకాల ఏజెంట్ మెమొరీలను మరియు వాటిని Microsoft Agent Frameworkతో ఎలా అమలు చేయాలో నేర్చుకున్నారు:

| మెమొరీ రకం | MAF మెకానిజం | జీవనకాలం |
|---|---|---|
| **కార్యదర్శి** | `agent.create_session()` | ఒక సంభాషణకు మాత్రమే |
| **తక్షణీయ** | ఒక థ్రెడ్‌లో సమీకరించబడిన సందర్భం | ఒక పని / సెషన్‌కి మాత్రమే |
| **దీర్ఘకాలిక** | `@tool` ఫంక్షన్ల ద్వారా యాక్సెస్ చేయబడే బాహ్య నిల్వ | సెషన్ల మధ్య |

### ముఖ్యమైన పాయింట్లు
1. **`agent.create_session()`** కార్యదర్శి మెమొరీని అందిస్తుంది — ఏజెంట్ సెషన్‌లో మొత్తం సంభాషణ చరిత్రను చూస్తుంది.
2. **కొత్త సెషన్లు సందర్భాన్ని కోల్పోతాయి** — దీర్ఘకాలిక మెమొరీ లేకుండా ఏజెంట్ గత సంభాషణలను గుర్తించలేడు.
3. **`@tool` ఫంక్షన్లు** అంతరాలను జత చేస్తాయి — అవి ఏజెంట్‌కు సమాచారం నిల్వ చేయడానికి మరియు పునఃప్రాప్తి చేసుకోవడానికి వీలు కల్పిస్తాయి.
4. **వ్యక్తిగతీకరణ సమయం కంటే మెరుగవుతుంది** — ఎక్కువ ప్రాధాన్యతలను నిల్వ చేసినప్పుడల్లా ఏజెంట్ సూచనలు మెరుగవుతాయి.

### వాస్తవ ప్రపంచ అన్వయాలు
- **గ్రాహక సేవ**: కస్టమర్ చరిత్ర మరియు ఇష్టాలను గుర్తుంచుకోవడం
- **వ్యక్తిగత సహాయకులు**: రోజులు లేదా వారాల పాటు సందర్భాన్ని సూచించడం
- **ఆరోగ్య సంరక్షణ**: రోగి సమాచారాన్ని మరియు ఇష్టాలను ట్రాక్ చేయడం
- **ఇ-కామర్స్**: చరిత్ర ఆధారంగా వ్యక్తిగత షాపింగ్

### తదుపరి జాబితా
- in-memory డిక్ట్‌ని డేటాబేస్ లేదా వెక్టర్ స్టోర్‌తో మార్చడం (ఉదా: Azure AI Search)
- సమయ సున్నితమైన సమాచారం కోసం మెమొరీ గడువు వేయడం
- షేరైడ్ మెమొరీతో బహుళ ఏజెంట్ వ్యవస్థలను నిర్మించడం
- జ్ఞాన-గ్రాఫ్ మద్దతుతో Cognee నోట్బుక్‌ని అన్వేషించడం


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**విడుదల సమాచారం**:  
ఈ డాక్యుమెంట్‌ను AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము సరిగ్గా ఇవ్వడానికి ప్రయత్నించినప్పటికీ, ఆటోమేటెడ్ అనువాదాల్లో పొరపాట్లు లేదా తప్పుడు వివరాలు ఉండవచ్చు. మూల పత్రం తన స్వదేశ భాషలోనే అధికారిక స్రవంతి అని భావించాలి. ముఖ్యమైన సమాచారానికి, ఒక నిపుణుల మానవ అనువాదాన్ని సూచిస్తాము. ఈ అనువాదం వాడకంవల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదోవల గురించి మేము బాధ్యులు కాదు.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
